https://youtu.be/mWRGwcMjFJM?si=UuOeysVRD0ohUlZE

In [0]:
data = [
    ("2010-01-02", 500),
    ("2010-02-03", 1000),
    ("2010-03-04", 1000),
    ("2010-04-05", 1000),
    ("2010-05-06", 1500),
    ("2010-06-07", 1000),
    ("2010-07-08", 1000),
    ("2010-08-09", 1000),
    ("2011-10-10", 1000),
    ("2011-01-02", 500),
    ("2011-02-03", 1000),
    ("2011-03-04", 1000),
    ("2011-04-05", 1000),
    ("2011-05-06", 1550),
    ("2011-06-07", 1100),
    ("2011-07-08", 1100),
    ("2011-08-09", 1000),
]

schema = ['date','sales']

df = spark.createDataFrame(data,schema)

df.display()

date,sales
2010-01-02,500
2010-02-03,1000
2010-03-04,1000
2010-04-05,1000
2010-05-06,1500
2010-06-07,1000
2010-07-08,1000
2010-08-09,1000
2011-10-10,1000
2011-01-02,500


In [0]:
from pyspark.sql import functions as F

In [0]:
df2 = (
    df.withColumns({
    "year" : F.year(F.col("date")),
    "quater" : F.quarter(F.col("date"))
}).groupBy("year","quater")
    .agg(F.sum("sales").alias("total_sum"))
    )

df2.display()

year,quater,total_sum
2010,1,2500
2010,2,3500
2010,3,2000
2011,4,1000
2011,1,2500
2011,2,3650
2011,3,2100


In [0]:
df3 = (
    df2.groupBy("year")
        .pivot("quater",[1,2])
        .agg(F.sum("total_sum"))
        .withColumnRenamed("1","Q1_sales")
        .withColumnRenamed("2","Q2_sales")
)
df3.display()

year,Q1_sales,Q2_sales
2010,2500,3500
2011,2500,3650


In [0]:
df4 = (
    df3.withColumn("percentage", 
                   F.when(F.col("Q1_sales").isNotNull() & F.col("Q2_sales").isNotNull(), 
                        ((F.col("Q2_sales") - F.col("Q1_sales")) / F.col("Q1_sales")) * 100))
)

df4.display()

year,Q1_sales,Q2_sales,percentage
2010,2500,3500,40.0
2011,2500,3650,46.0
